# Data Validation: Quality Checks for Production Pipelines

## Learning Objectives
- Implement layered validation (schema, completeness, statistical)
- Design data contracts between producers and consumers
- Detect and respond to data corruption
- Balance validation performance with accuracy

## Interview Questions This Covers
- "Design a data validation strategy for 1M transactions/day"
- "Data validation blocked 5% of data. Allow it or investigate?"
- "How do you implement data contracts?"
- "Model accuracy dropped but validation passed. What went wrong?"

## Basic Implementation: Layered Validation

In [ ]:
import pandas as pd
import numpy as np
from typing import Dict, List, Tuple
from dataclasses import dataclass

@dataclass
class ValidationResult:
    layer: str
    passed: bool
    records_failed: int
    failed_percentage: float
    details: str

class DataValidator:
    """Multi-layer data validation"""
    
    def __init__(self):
        self.schema = {}  # column -> type
        self.results = []
    
    def set_schema(self, schema: Dict[str, str]):
        """Define expected schema"""
        self.schema = schema
    
    def validate_schema(self, df: pd.DataFrame) -> ValidationResult:
        """Layer 1: Schema validation (type checks)"""
        failed = 0
        
        for col, expected_type in self.schema.items():
            if col not in df.columns:
                failed += len(df)
                continue
            
            # Check type
            if expected_type == 'int' and not pd.api.types.is_integer_dtype(df[col]):
                failed += (df[col].dtype != 'int64').sum()
            elif expected_type == 'float' and not pd.api.types.is_float_dtype(df[col]):
                failed += (df[col].dtype != 'float64').sum()
            elif expected_type == 'string' and df[col].dtype != 'object':
                failed += (df[col].dtype != 'object').sum()
        
        result = ValidationResult(
            layer='schema',
            passed=failed == 0,
            records_failed=failed,
            failed_percentage=100 * failed / len(df),
            details=f"Type mismatch in {failed} records"
        )
        self.results.append(result)
        return result
    
    def validate_completeness(self, df: pd.DataFrame, 
                             required_cols: List[str],
                             null_threshold: float = 0.01) -> ValidationResult:
        """Layer 2: Completeness validation"""
        failed = 0
        
        for col in required_cols:
            null_count = df[col].isnull().sum()
            null_pct = null_count / len(df)
            
            if null_pct > null_threshold:
                failed += null_count
        
        result = ValidationResult(
            layer='completeness',
            passed=failed == 0,
            records_failed=failed,
            failed_percentage=100 * failed / len(df),
            details=f"Nulls in required columns: {failed} records"
        )
        self.results.append(result)
        return result
    
    def validate_statistical(self, df: pd.DataFrame,
                           col: str,
                           expected_range: Tuple[float, float],
                           outlier_threshold: float = 5.0) -> ValidationResult:
        """Layer 3: Statistical validation"""
        failed = 0
        
        # Check range
        out_of_range = (df[col] < expected_range[0]) | (df[col] > expected_range[1])
        failed += out_of_range.sum()
        
        # Check outliers (values beyond outlier_threshold std devs)
        mean = df[col].mean()
        std = df[col].std()
        outliers = np.abs(df[col] - mean) > outlier_threshold * std
        
        result = ValidationResult(
            layer='statistical',
            passed=failed == 0 and outliers.sum() == 0,
            records_failed=failed + outliers.sum(),
            failed_percentage=100 * (failed + outliers.sum()) / len(df),
            details=f"Out of range: {failed}, Outliers: {outliers.sum()}"
        )
        self.results.append(result)
        return result
    
    def validate_business_logic(self, df: pd.DataFrame,
                               rule_func) -> ValidationResult:
        """Layer 4: Business logic validation"""
        failed = (~rule_func(df)).sum()
        
        result = ValidationResult(
            layer='business_logic',
            passed=failed == 0,
            records_failed=failed,
            failed_percentage=100 * failed / len(df),
            details=f"Business rule violations: {failed} records"
        )
        self.results.append(result)
        return result

# Usage
validator = DataValidator()

# Define schema
validator.set_schema({
    'user_id': 'int',
    'amount': 'float',
    'timestamp': 'string',
    'merchant_id': 'int'
})

# Create sample transaction data
df = pd.DataFrame({
    'user_id': [1, 2, 3, 4, 5],
    'amount': [100.0, 50.0, 200.0, -10.0, 150.0],  # One negative (invalid)
    'timestamp': ['2026-05-16', '2026-05-16', '2026-05-16', '2026-05-16', None],  # One null
    'merchant_id': [101, 102, 103, 104, 105]
})

# Run validation layers
print("Running validation layers:")
print()

r1 = validator.validate_schema(df)
print(f"✓ Layer 1 (Schema): {'PASS' if r1.passed else 'FAIL'} - {r1.details}")

r2 = validator.validate_completeness(df, required_cols=['timestamp'], null_threshold=0.01)
print(f"✓ Layer 2 (Completeness): {'PASS' if r2.passed else 'FAIL'} - {r2.details}")

r3 = validator.validate_statistical(df, col='amount', expected_range=(0, 10000))
print(f"✓ Layer 3 (Statistical): {'PASS' if r3.passed else 'FAIL'} - {r3.details}")

r4 = validator.validate_business_logic(df, lambda x: x['amount'] > 0)
print(f"✓ Layer 4 (Business Logic): {'PASS' if r4.passed else 'FAIL'} - {r4.details}")

print()
print("Overall: FAIL - Bad records detected in layers 2, 3, 4")

## Advanced Implementation: Data Contracts

In [ ]:
from dataclasses import dataclass
from datetime import datetime, timedelta
from typing import Optional

@dataclass
class DataContract:
    """Formal agreement between data producer and consumer"""
    dataset_name: str
    owner: str
    schema: Dict[str, str]  # column -> type
    freshness_sla_hours: int  # data must be updated within this time
    completeness_threshold: float  # % of required fields must be present
    value_ranges: Dict[str, Tuple[float, float]]  # acceptable value ranges
    version: str

class ContractValidator:
    """Validate data against contract"""
    
    def __init__(self):
        self.contracts = {}
        self.violations = []
    
    def register_contract(self, contract: DataContract):
        """Register a data contract"""
        self.contracts[contract.dataset_name] = contract
        print(f"✓ Registered contract for {contract.dataset_name} (v{contract.version}) by {contract.owner}")
    
    def validate_against_contract(self, df: pd.DataFrame, 
                                  dataset_name: str,
                                  last_update: datetime) -> Dict[str, any]:
        """Check data against contract"""
        contract = self.contracts[dataset_name]
        violations = []
        
        # Check freshness
        age_hours = (datetime.now() - last_update).total_seconds() / 3600
        if age_hours > contract.freshness_sla_hours:
            violations.append(f"Freshness SLA violated: {age_hours:.1f}h old (SLA: {contract.freshness_sla_hours}h)")
        
        # Check schema
        for col, dtype in contract.schema.items():
            if col not in df.columns:
                violations.append(f"Missing column: {col}")
        
        # Check completeness
        completeness = 1 - (df.isnull().sum().sum() / (len(df) * len(contract.schema)))
        if completeness < contract.completeness_threshold:
            violations.append(f"Completeness: {completeness:.1%} (threshold: {contract.completeness_threshold:.1%})")
        
        # Check value ranges
        for col, (min_val, max_val) in contract.value_ranges.items():
            if col in df.columns:
                out_of_range = ((df[col] < min_val) | (df[col] > max_val)).sum()
                if out_of_range > 0:
                    violations.append(f"Out of range ({col}): {out_of_range} records")
        
        return {
            'dataset': dataset_name,
            'contract_version': contract.version,
            'validated': True,
            'passed': len(violations) == 0,
            'violations': violations,
        }

# Usage
cv = ContractValidator()

# Register contract: "user_spend" dataset
spend_contract = DataContract(
    dataset_name='user_spend',
    owner='payments_team',
    schema={'user_id': 'int', 'amount': 'float', 'date': 'string'},
    freshness_sla_hours=24,
    completeness_threshold=0.99,
    value_ranges={'amount': (0, 100000)},
    version='v1'
)
cv.register_contract(spend_contract)

# Create data
spend_df = pd.DataFrame({
    'user_id': [1, 2, 3, 4, 5],
    'amount': [50.0, 200.0, 150.0, 100.0, 75.0],
    'date': ['2026-05-16', '2026-05-16', '2026-05-16', '2026-05-16', '2026-05-16']
})

# Validate
print()
result = cv.validate_against_contract(
    spend_df,
    'user_spend',
    last_update=datetime.now() - timedelta(hours=2)  # 2 hours old
)

print(f"\nContract Validation Result:")
print(f"  Dataset: {result['dataset']}")
print(f"  Contract: v{result['contract_version']}")
print(f"  Status: {'✓ PASS' if result['passed'] else '✗ FAIL'}")
if result['violations']:
    print(f"  Violations:")
    for v in result['violations']:
        print(f"    - {v}")

## Real-World Example 1: Netflix Validation

In [ ]:
import pandas as pd
import numpy as np

def netflix_metadata_validation():
    """Validate 7K+ titles with 1000+ features"""

    print("NETFLIX: Content Metadata Validation")
    print("=" * 60)

    # Simulate Netflix catalog
    np.random.seed(42)
    titles = pd.DataFrame({
        'title_id': range(1, 7001),
        'title': [f'Show_{i}' for i in range(7000)],
        'country': np.random.choice(['US', 'UK', 'FR', 'DE', 'JP'], 7000),
        'rating': np.random.uniform(1, 5, 7000),
        'budget': np.random.lognormal(10, 2, 7000),
        'release_year': np.random.randint(1990, 2026, 7000),
    })

    # Introduce errors (realistic)
    titles.loc[titles.sample(15).index, 'rating'] = 9.5  # invalid rating
    titles.loc[titles.sample(20).index, 'budget'] = -1000  # negative budget
    titles.loc[titles.sample(10).index, 'country'] = 'USA'  # wrong code
    titles.loc[titles.sample(5).index, 'release_year'] = 2050  # future date

    print(f"Total titles: {len(titles):,}\n")

    # Validation 1: Rating range (1-5)
    valid_rating = (titles['rating'] >= 1) & (titles['rating'] <= 5)
    print(f"VALIDATION 1 - Rating (1-5):")
    print(f"  Invalid: {(~valid_rating).sum()}")
    print(f"  Invalid values: {titles[~valid_rating]['rating'].unique()}")
    print()

    # Validation 2: Budget > 0
    valid_budget = titles['budget'] > 0
    print(f"VALIDATION 2 - Budget (> 0):")
    print(f"  Invalid: {(~valid_budget).sum()}")
    print()

    # Validation 3: Country code (valid ISO)
    valid_countries = titles['country'].isin(['US', 'UK', 'FR', 'DE', 'JP'])
    print(f"VALIDATION 3 - Country Code:")
    print(f"  Invalid: {(~valid_countries).sum()}")
    print(f"  Invalid codes: {titles[~valid_countries]['country'].unique()}")
    print()

    # Validation 4: Release year (past dates)
    valid_year = titles['release_year'] <= 2025
    print(f"VALIDATION 4 - Release Year (≤ 2025):")
    print(f"  Invalid: {(~valid_year).sum()}")
    print()

    # Clean dataset
    clean = titles[valid_rating & valid_budget & valid_countries & valid_year]
    print(f"FINAL RESULT:")
    print(f"  Valid titles: {len(clean):,}")
    print(f"  Removed: {len(titles) - len(clean):,}")
    print(f"  Retention rate: {len(clean)/len(titles)*100:.1f}%")

netflix_metadata_validation()


## Real-World Example 2: Stripe Transaction Validation

In [ ]:
import pandas as pd
import numpy as np

def stripe_transaction_validation():
    """Validate 500M+ transactions/day"""

    print("STRIPE: Transaction Data Validation")
    print("=" * 60)

    # Simulate transactions
    np.random.seed(42)
    n_txns = 100_000  # sample of 500M/day

    # Clean transactions (99%)
    clean_txns = pd.DataFrame({
        'amount': np.random.lognormal(4.5, 1.2, int(n_txns * 0.99)),
        'currency': np.random.choice(['USD', 'EUR', 'GBP'], int(n_txns * 0.99)),
        'merchant_id': np.random.randint(1, 100000, int(n_txns * 0.99)),
    })

    # Corrupted transactions (1%)
    corrupt_txns = pd.DataFrame({
        'amount': np.concatenate([
            [-50] * 250,  # negative amounts
            [999999] * 250,  # outlier amounts
            [0] * 250,  # zero amounts
        ]),
        'currency': ['INVALID'] * 750,
        'merchant_id': [0] * 750,
    })

    # Combine
    df = pd.concat([clean_txns, corrupt_txns], ignore_index=True)

    print(f"Total transactions: {len(df):,}")
    print()

    # Validation 1: Amount range
    valid_amount = (df['amount'] > 0) & (df['amount'] < 999999)
    print(f"VALIDATION 1 - Amount Range (0, 999999):")
    print(f"  Valid: {valid_amount.sum():,}")
    print(f"  Invalid: {(~valid_amount).sum():,}")
    print()

    # Validation 2: Currency
    valid_currencies = df['currency'].isin(['USD', 'EUR', 'GBP'])
    print(f"VALIDATION 2 - Currency (USD, EUR, GBP):")
    print(f"  Valid: {valid_currencies.sum():,}")
    print(f"  Invalid: {(~valid_currencies).sum():,}")
    if (~valid_currencies).sum() > 0:
        print(f"  Found: {df[~valid_currencies]['currency'].unique()}")
    print()

    # Validation 3: Merchant ID
    valid_merchant = df['merchant_id'] > 0
    print(f"VALIDATION 3 - Merchant ID (> 0):")
    print(f"  Valid: {valid_merchant.sum():,}")
    print(f"  Invalid: {(~valid_merchant).sum():,}")
    print()

    # Final clean dataset
    clean = df[valid_amount & valid_currencies & valid_merchant]
    print(f"FINAL RESULT:")
    print(f"  Clean transactions: {len(clean):,}")
    print(f"  Removed: {len(df) - len(clean):,} ({(len(df)-len(clean))/len(df)*100:.2f}%)")
    print(f"  Status: Ready for fraud model training ✓")

stripe_transaction_validation()


## Real-World Example 3: Uber Ride Validation

In [ ]:
def uber_validation():
    print("Uber: Ride Data Validation")
    print()
    
    print("1. Schema:")
    print("   - driver_id, rider_id, distance (km), duration (min), fare")
    print()
    
    print("2. Completeness:")
    print("   - All fields required (no nulls)")
    print()
    
    print("3. Statistical:")
    print("   - distance: [0.1, 100] km")
    print("   - duration: [2, 180] minutes")
    print("   - fare: [$2, $500]")
    print()
    
    print("4. Business Logic (Sanity Check):")
    print("   - duration > distance / 60 (minimum speed: 60 km/h)")
    print("   - fare >= distance * $0.5 (minimum: $0.50/km)")
    print("   - fare <= distance * $3 (maximum: $3/km)")
    print()
    
    print("5. Anomaly Detection:")
    print("   - 10x longer than typical = alert")
    print("   - 5x more expensive than typical = investigate")
    print()
    
    print("6. Pipeline:")
    print("   - Real-time: validate each ride immediately")
    print("   - Failure: block ride from analytics, alert team")
    print("   - Success: feed to pricing & ETA models")

uber_validation()

## Interview Case Study: Financial Services Validation

**Scenario:** Design a data validation strategy for a financial services company processing 10M transactions/day.

In [ ]:
print("INTERVIEW CASE STUDY: FINANCIAL DATA VALIDATION")
print()

print("CONSTRAINTS:")
print("  - 10M+ transactions per day")
  - Real-time fraud detection requires fresh data")
print("  - Regulatory compliance (GDPR, PCI)")
print("  - False positives = customer frustration (declined legitimate txns)")
print("  - False negatives = fraud losses")
print()

print("SOLUTION: LAYERED VALIDATION STRATEGY")
print()

print("Layer 1 - Schema (Fast, Strict):")
print("  ✓ Run on raw data before any processing")
print("  ✓ Check types: amount=float, user_id=string, timestamp=datetime")
print("  ✓ Reject on type mismatch (0% tolerance)")
print("  ✓ Latency: 10ms for 100K records")
print()

print("Layer 2 - Completeness (Fast, Moderate):")
print("  ✓ Check required fields: user_id, amount, merchant_id, timestamp")
print("  ✓ Threshold: 99.9% completeness (0.1% nulls acceptable)")
print("  ✓ If <99.9%: alert, investigate source")
print("  ✓ Latency: 5ms per 100K records")
print()

print("Layer 3 - Statistical (Moderate, Smart):")
print("  ✓ Amount: [0.01, 10,000]")
print("  ✓ Median $50, p95 $500 (from historical)")
print("  ✓ Flag if 10x outliers detected")
print("  ✓ Latency: 50ms per 100K records (compute percentiles)")
print()

print("Layer 4 - Business Logic (Smart, Risk-Aware):")
print("  ✓ Merchant-specific rules: typical spend pattern")
print("  ✓ User-specific rules: usual amount range")
print("  ✓ Regulatory: KYC (Know Your Customer) checks")
print("  ✓ Latency: 100ms per 100K records (includes DB lookups)")
print()

print("OPERATIONAL DECISIONS:")
print()
print("  1. Parallelization:")
print("     - Run Layers 1-2 in parallel (don't block on either)")
print("     - Run Layer 3 separately (more expensive)")
print("     - Run Layer 4 async (can delay by 1 minute)")
print()

print("  2. Failure Handling:")
print("     - Layer 1 fail: REJECT transaction immediately")
print("     - Layer 2 fail: ALERT on data source issue")
print("     - Layer 3 fail: FLAG for investigation (don't auto-block)")
print("     - Layer 4 fail: CHALLENGE user (require verification)")
print()

print("  3. Monitoring:")
print("     - Track % passing each layer")
print("     - Alert if >1% fail Layer 1 (schema issue)")
print("     - Alert if >5% fail Layer 3 (distribution shift)")
print("     - Track false positive rate from Layer 4")
print()

print("RESULTS:")
print("  Latency: 10 + 5 + 50 = 65ms (before async layer 4)")
print("  Throughput: 10M/day = 115 txns/sec, easily achievable")
print("  Coverage: Catches 95%+ of data quality issues")
print("  False positive rate: <1% (customer experience)")

## Key Takeaways

**2-Minute Elevator Pitch:**
"Data validation prevents garbage data from training models or serving predictions. Implement layers: schema (strict), completeness (moderate), statistical (smart), business logic (risk-aware). Data contracts formalize expectations between producers and consumers. Fail loud when data breaks; investigate root causes. Monitor validation metrics to catch issues early."

**When to Investigate vs Allow:**
- Layer 1/2 failures: Always investigate (data is broken)
- Layer 3 failures: Investigate if >5% (distribution shift)
- Layer 4 failures: Smart response (challenge, block, or allow)

**Common Follow-Ups:**
- "Data validation blocked 5% of data" → Investigate if real issue or false positive
- "Model accuracy dropped but validation passed" → Need smarter statistical checks
- "Validation is too slow" → Parallelize, sample, or move to async
- "Too many false positives" → Loosen thresholds, add context